# 音声合成モデルの比較ノートブック

**目的**：同じ文章を Style-Bert-VITS2（JVNV・小春音アミ）と Kokoro で合成し、書き起こし誤り率・速度・参照なし品質推定を比べる。
ローカルで実行した結果は `reports/audio_comparison.md` にある。

- 依存関係：`style-bert-vits2`, `kokoro-onnx`, `misaki[ja]`, `fugashi`, `unidic-lite`, `faster-whisper`, `torchaudio`, `pyworld`
- 入力：HuggingFace のモデル（初回に自動でダウンロード、約3 GB）／出力：`audio/comparison/*.wav`, `reports/audio_comparison*.{md,csv}`
- **GPU が使えない場合**：そのまま CPU で動く（12声×8文で約12分）
- **ライセンス**：JVNV モデルは CC BY-SA 4.0、小春音アミ・あみたろはクレジット表記が必要（README 参照）。生成音声を実在の人物の声として扱わない
- **注意**：`unidic`（フル版）が入っていると MeCab の初期化に失敗する。`unidic-lite` だけを使う

In [ ]:
import importlib.util


def ensure(pkgs: dict) -> None:
    """入っていないパッケージだけを入れる（pip がなければ uv を使う）。フル版の unidic は MeCab の初期化に失敗するので外す。"""
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    has_pip = importlib.util.find_spec("pip") is not None
    if missing:
        cmd = [sys.executable, "-m", "pip", "install", "-q"] if has_pip else ["uv", "pip", "install", "--python", sys.executable]
        subprocess.run(cmd + missing, check=True)
    if importlib.util.find_spec("unidic") is not None:
        cmd = [sys.executable, "-m", "pip", "uninstall", "-y", "-q"] if has_pip else ["uv", "pip", "uninstall", "--python", sys.executable]
        subprocess.run(cmd + ["unidic"], check=True)
    print("追加したパッケージ:", missing or "なし")

import os, sys, subprocess, shutil, platform
from pathlib import Path
ON_KAGGLE = Path("/kaggle").exists()
# プロジェクトの場所：Kaggle では Dataset として追加した production/gene-quest を /kaggle/working にコピーして使う
PROJECT = Path(os.environ.get("GQ_PROJECT", "/kaggle/working/gene-quest" if ON_KAGGLE else Path.cwd().parent))
SRC = Path("/kaggle/input/gene-quest") if ON_KAGGLE else None
if ON_KAGGLE and not PROJECT.exists() and SRC and SRC.exists():
    shutil.copytree(SRC, PROJECT)
print("project:", PROJECT, "exists:", PROJECT.exists())

In [ ]:
ensure({"style_bert_vits2": "style-bert-vits2", "kokoro_onnx": "kokoro-onnx", "misaki": "misaki[ja]", "fugashi": "fugashi",
        "unidic_lite": "unidic-lite", "faster_whisper": "faster-whisper", "pyloudnorm": "pyloudnorm", "soundfile": "soundfile",
        "yaml": "pyyaml", "jaconv": "jaconv"})

In [ ]:
# モデルのダウンロード（済んでいれば飛ばす）
subprocess.run(["bash", str(PROJECT / "scripts" / "download_models.sh")], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/compare_tts.py"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "scripts/score_tts_quality.py"], cwd=PROJECT)
print((PROJECT / "reports" / "audio_comparison.md").read_text())